In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import pymc as pm
import arviz as az
import pandas as pd
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import Optional
import random
import itertools
import warnings

# Notebook config
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

# Board Game Simulation

## Volcano Rush

### Abstract

*(To be filled at the end of the project)*

- Summary of simulation approach and scope
- Key findings: win rates, character balance, resource dynamics
- Main conclusions on game design

### Introduction

Game balance is considered critical to the success of any game, yet there is no universal consensus on what it actually means [1]. It can be broadly interpreted as the process of tuning a game's rules, difficulty, play time, resources, and role abilities so that no single strategy, character, or configuration dominates the experience. For multiplayer games in particular, balance has several dimensions: ensuring no starting position is inherently advantageous, preventing any strategy from being strictly dominant, and calibrating the cost-to-benefit ratio across different game objects [1].

As a semi-cooperative board game, *Volcano Rush* faces a specific balancing challenge: the group must share a win condition (escaping the volcano), while players also compete for individual scores. A well-balanced game therefore needs a group win rate that feels achievable but not trivial, and individual scores that are not dominated by a single character role. Compounding this, the game supports 4 to 8 players, meaning balance must hold across a wide range of player configurations.

As with most game design problems, there is no deterministic recipe for achieving balance. *Volcano Rush* is both strategic and social, and raw numbers can sometimes mislead. Simulation is not a substitute for playtesting, but it provides a principled starting point for identifying where imbalances may arise before bringing the game to the table.

### Related Work

#### Monte Carlo simulation

Monte Carlo simulation estimates probabilities by running many random trials instead of solving mathematical formulas, which can be extremely difficult for complex games. Running $n$ simulated games under a given strategy $M$ yields an empirical win probability:

We define an indicator variable for each simulation:

- $X_i = 1$ if simulation $i$ results in a win  
- $X_i = 0$ otherwise  

The empirical win probability after $n$ simulations is:

$$
\hat{p}_n(M) = \frac{1}{n} \sum_{i=1}^{n} X_i
$$

As the number of simulations increases, this estimate converges to the true win probability:

$$
p(M) = \lim_{n \to \infty} \frac{1}{n} \sum_{i=1}^{n} X_i
$$

This follows from the **Law of Large Numbers**, which states that the sample average converges to the expected value (true probability) as the number of trials grows. This makes Monte Carlo methods a good fit for games with many possible situations, where computing the exact answer is not practical.

#### Monte Carlo Tree Search (MCTS)

Monte Carlo methods are also widely used in game AI through Monte Carlo Tree Search (MCTS) [2], which builds a tree of possible future game states and tests them with simulations. 
At each step, the algorithm faces a trade-off between two ideas:
- *exploitation* — choosing moves that have already shown good results  
- *exploration* — trying moves that have not been tested much but might be better  

This balance is handled by the **Upper Confidence Bound for Trees** (UCT) formula:

$$ \mathrm{UCT} = \frac{W_i}{N_i} + c \sqrt{\frac{\ln N_p}{N_i}} $$

The formula combines two parts:
- $\frac{W_i}{N_i}$ — the **exploitation term**, which represents how successful a move has been so far (its average win rate)
- $c \sqrt{\frac{\ln N_p}{N_i}}$ — the **exploration term**, which gives a higher value to moves that have been tried fewer times, encouraging the algorithm to explore them
- $c$ — a constant that controls the balance between exploration and exploitation (higher values lead to more exploration)

In practice, MCTS does not explore all possible game paths. Instead, it gradually builds the search tree through repeated simulations. At the beginning, it gathers initial data by trying different possible moves. As more simulations are performed, the algorithm starts to focus more on the moves that show better results, exploring them in greater depth. However, less-explored moves are still occasionally tested due to the exploration term in the UCT formula, ensuring that potentially good alternatives are not missed.

In this work, however, we use simpler repeated simulations with rule-based agents to study game balance rather than to improve decision-making [2].

#### Agent-Based Modeling (ABM)

In Agent-Based Modeling (ABM), each agent follows a set of simple rules and interacts with other agents and the world around it [3]. Over time, these small interactions can produce complex group behaviors that would be hard to predict just by looking at the rules alone.

Compared to pure Monte Carlo simulation, ABM allows for more realistic behavior. Monte Carlo is fully random, but in ABM, agents can follow rules that reflect real human decisions. This approach has been used in research: Deadman, Schlager, and Gimblett (2000) interviewed participants after a game experiment and used their answers to design the decision rules for ABM agents [5]. For *Volcano Rush*, we could do the same — interview players after a playtest session and use what we learn to make the agents behave more like real people [4].

Traditional game theory assumes that players always make the best possible decision to get the best outcome for themselves. But in real life, people are often influenced by emotions, habits, and not having all the information [3]. This means game theory models do not always reflect how real people actually play.

### Data

- No external dataset — all data is synthetically generated by the simulation
- Game parameter sources:
  - Mission pool (8 standard + 5 boat missions) from `docs/game_rules.md`
  - Complication deck (11 cards) — draw probabilities derived from deck composition
  - Volcano deck (11 cards + Eruption at bottom)
  - Character roles (6) with defined abilities
  - Player count configurations: 4, 5, 6, 7, 8
- Simulation assumptions documented in `docs/simulation_assumptions.md` (exhaustion timing, Craftsman repair cycle, Sailor severity scoring, etc.)
- Each simulation run = one full game; N runs per configuration = our dataset
- Derived output variables: win/loss, rounds elapsed, individual scores, boat parts completed, volcano cards remaining

### Methods

- Monte Carlo simulation: run thousands of games under fixed strategy assumptions
- Agent strategy modeled as rule-based heuristics — e.g., always participate if resources allow
- Sensitivity analysis: vary strategy aggressiveness, player count, character assignment
- Bayesian inference (PyMC) to estimate win-rate distributions with uncertainty

#### Define Game State & Parameters

- Dataclasses / enums for: `Player`, `Character`, `Mission`, `ComplicationCard`, `VolcanoCard`, `GameState`
- Parameters: player count, character assignments, deck compositions, resource deck size
- Strategy parameter: participation threshold (minimum resources a player needs to decide to join a mission)

In [1]:
# Game state & data structure definitions


#### Implement Simulation Engine

- Round loop: 6 steps (mission selection → participation → complication draw → resolution → exhaustion → mission refresh)
- Mission resolution: count participants, check tool availability, apply character abilities, check success
- Handle special complications (Night Anxiety requires a non-participant helper, Camp Panic limits resources per player, etc.)
- Handle volcano effects (Rain and Mud cost increases, Heat Wave extended exhaustion, Lava Flow removes missions, etc.)
- Terminal conditions: Eruption card drawn (loss) or all required boat missions complete (win)

In [2]:
# Simulation engine


#### Configure Simulation Scenarios

- Scenarios by player count: 4, 5, 6, 7, 8
- Character composition: random assignment vs. fixed "strong" team vs. fixed "weak" team
- Strategy variants: aggressive (always participate) vs. conservative (gather-heavy)
- Number of runs per scenario: N = 1,000–10,000 (tune based on variance)

In [3]:
# Scenario configuration & run loop


#### Exploratory Simulation Runs

- Single-game trace to verify logic (print round-by-round state)
- Check distribution of game lengths (rounds to win or lose)
- Sanity checks: resource deck exhaustion rate, average volcano cards drawn per game

In [4]:
# Exploratory runs + sanity checks


#### Visualize Outcomes

- Win rate by player count (bar chart with confidence intervals)
- Distribution of game length (histogram)
- Character score distributions (violin or box plot per character)
- Volcano deck depth at game end (how close to eruption)
- Resource economy: average cards in hand per round
- Mission completion heatmap (which missions complete most often)

In [5]:
# Plots


#### Statistical Analysis

- Bayesian estimation of win rate per scenario using PyMC Beta-Binomial model
- Posterior comparison: does player count 6 win significantly more often than 4?
- Character dominance: Bayesian comparison of individual scores per character
- ArviZ for posterior visualization and credible intervals

In [6]:
import pymc as pm
import arviz as az

# PyMC models + ArviZ plots

### Results

- Win rates across player counts — is it easier with more players?
- Which characters emerge as score leaders? Is there a dominant strategy?
- Resource bottlenecks: which resource type runs out most often?
- How often does the volcano erupt vs. the group escaping?
- Boat mission difficulty ranking (which cause the most failures?)
- Complication card impact — which ones most often flip a win to a loss?

### Discussion / Further Work

- Is the game balanced as designed, or does it favor a specific player count?
- Suggestions for rule tweaks to improve balance (adjust required boat parts, resource deck size)
- Limitations: heuristic strategies don't capture human creativity or negotiation
- Next steps:
  - Replace rule-based agents with learned strategies (RL or MCTS)
  - Explore asymmetric information (players don't see each other's hands)
  - Sensitivity to rule variants (e.g., exhaustion only on success)
  - Compare simulated win rates against real playtest results

### References / Bibliography

[1] [Schell, J. (2009). Level 16: Game Balance. Game Design Concepts.](https://gamedesignconcepts.wordpress.com/2009/08/20/level-16-game-balance/)

[2] [Chaslot, G., Bakkes, S., Szita, I., & Spronck, P. (2008). Monte-Carlo Tree Search: A New Framework for Game AI. AIIDE.](https://sander.landofsand.com/publications/AIIDE08_Chaslot.pdf)

[3] [StudyGuides.com. (2026). Simulations in Game Theory.](https://studyguides.com/study-methods/overview/clzggcm6j0j8t8cfe9zugo2t8)

[4] [Dubois, E., Barreteau, O., & Souchère, V. (2013). An Agent-Based Model to Explore Game Setting Effects on Attitude Change During a Role Playing Game Session. Journal of Artificial Societies and Social Simulation, 16(1), 2.](https://www.jasss.org/16/1/2.html)

[5] [Yiannakoulias, N., Grignon, M., & Marshall, T. (2024). Parameterizing agent-based models using an online game. Computers, Environment and Urban Systems, 112, 102142.](https://www.sciencedirect.com/science/article/pii/S0198971524000711)

### Appendix

- **Appendix A:** Full mission table (from `docs/game_rules.md`)
- **Appendix B:** Complication card descriptions
- **Appendix C:** Volcano card descriptions
- **Appendix D:** Character ability reference table
- **Appendix E:** Simulation assumptions (cross-ref `docs/simulation_assumptions.md`)
- **Appendix F:** Raw simulation output tables